In [ ]:
# from dotenv import load_dotenv
# import os

# load_dotenv()
# hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")


In [2]:
!pip install langchain langchain-core langchain-community langchain-huggingface huggingface_hub chromadb faiss-cpu tiktoken wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.9 MB 1.7 MB/s eta 0:00:11
   -- ------------------------------------- 1.0/18.9 MB 2.1 MB/s eta 0:00:09
   --- ------------------------------------ 1.6/18.9 MB 2.1 MB/s eta 0:00:09
   ---- ----------------------------------- 2.1/18.9 MB 2.2 MB/s eta 0:00:08
   ---- ----------------------------------- 2.4/18.9 MB 2.2 MB/s eta 0:00:08
   ------ --------------------------------- 2.9/18.9 MB 2.3 MB/s eta 0:00:08
   ------ --------------------------------- 3.1/18.9 MB 2.3 MB/s eta


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\kanha\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


## Wikipedia Retriever

In [3]:
from langchain_community.retrievers import WikipediaRetriever

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [ ]:

# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [ ]:
docs

[Document(metadata={'title': 'India–Iran relations', 'summary': "The Republic of India and the Islamic Republic of Iran maintain a bilateral relationship. Independent India and Iran established diplomatic relations on 15 March 1950.\nContact between both ancient Persia and ancient India date to ancient times, and can be seen through the diffusion of Persian culture among Islamic culture in much of South Asia; furthermore, around 15% of the Muslims in India are Shia, a group Iran considers itself to represent on the world stage. Outside the Islamic community, the impact of Persian culture has primarily been in Northwest India.\nDuring much of the Cold War, relations between India and the erstwhile Imperial State of Iran suffered due to their differing political interests: India endorsed a non-aligned position but fostered strong links with the Soviet Union, while Iran was an open member of the Western Bloc and enjoyed close ties with the United States. While India did not welcome the 19

In [ ]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
The Republic of India and the Islamic Republic of Iran maintain a bilateral relationship. Independent India and Iran established diplomatic relations on 15 March 1950.
Contact between both ancient Persia and ancient India date to ancient times, and can be seen through the diffusion of Persian culture among Islamic culture in much of South Asia; furthermore, around 15% of the Muslims in India are Shia, a group Iran considers itself to represent on the world stage. Outside the Islamic community, the impact of Persian culture has primarily been in Northwest India.
During much of the Cold War, relations between India and the erstwhile Imperial State of Iran suffered due to their differing political interests: India endorsed a non-aligned position but fostered strong links with the Soviet Union, while Iran was an open member of the Western Bloc and enjoyed close ties with the United States. While India did not welcome the 1979 Islamic Revolution, relations between

## Vector Store Retriever

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [ ]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [ ]:
# Step 2: Initialize embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [ ]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [ ]:
results = vectorstore.similarity_search(query, k=2)

In [ ]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


## MMR 

In [ ]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize Hugging Face embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",                   
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)

In [ ]:
query = "What is langchain?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 2 ---
LangChain is used to build LLM based applications.

--- Result 3 ---
Embeddings are vector representations of text.


## ContextualCompressionRetriever

In [ ]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-huggingface

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.3.0
Uninstalling langchain-core-1.3.0:
  Successfully uninstalled langchain-core-1.3.0
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-huggingface 1.2.2
Uninstalling langchain-huggingface-1.2.2:
  Successfully uninstalled langchain-huggingface-1.2.2


In [ ]:
!pip install langchain langchain-core langchain-community langchain-huggingface

  Using cached langchain_core-1.3.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 5.3 MB/s eta 0:00:00
Using cached langchain_core-1.3.0-py3-none-any.whl (515 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever
import os

ModuleNotFoundError: No module named 'langchain.retrievers'

In [ ]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

NameError: name 'Document' is not defined